In [16]:
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted
import json
import os
import asyncio
from dotenv import load_dotenv

In [ ]:

load_dotenv()
my_api_key = os.environ.get("GEMINI_API_KEY")

if not my_api_key:
    raise ValueError("API key not found! Check your .env file.")

genai.configure(api_key=my_api_key)

# Using the correct model for extraction
model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

generation_config = genai.types.GenerationConfig(
    temperature=0.0, 
    response_mime_type="application/json",
)

system_instruction = """
You are a highly precise medical extraction AI. 
Find all diseases and chemicals in the provided text. 
Return ONLY a valid JSON object with two arrays: "diseases" and "chemicals".
If none are found, return empty arrays.
"""

In [ ]:
async def process_single_doc(doc, semaphore):
    async with semaphore: 
        full_prompt = f"{system_instruction}\n\nText to analyze: {doc['text']}"
        retry_count = 0
        
        while True:
            try:
                response = await model.generate_content_async(
                    full_prompt,
                    generation_config=generation_config
                )
                
                extracted_data = json.loads(response.text)
                print(f"   [✓] Success: Doc {doc['doc_id']}")
                return {
                    "doc_id": doc["doc_id"],
                    "gemini_response": extracted_data
                }
                
            except ResourceExhausted:
                retry_count += 1
                sleep_time = 2 * retry_count 
                print(f"   [!] Burst limit hit on {doc['doc_id']}. Retrying in {sleep_time}s...")
                await asyncio.sleep(sleep_time)
                
            except Exception as e:
                print(f"   [X] Fatal Error on doc {doc['doc_id']}: {e}")
                return {
                    "doc_id": doc["doc_id"],
                    "gemini_response": {"diseases": [], "chemicals": []}
                }

async def run_gemini_pipeline_async(input_filename, output_filename):
    print(f"Loading data from {input_filename}...")
    
    with open(input_filename, "r", encoding="utf-8") as infile:
        dataset = json.load(infile)
        
    semaphore = asyncio.Semaphore(20) 
    
    print(f"Firing {len(dataset)} requests asynchronously...")
    
    tasks = [process_single_doc(doc, semaphore) for doc in dataset]
    
    final_results = await asyncio.gather(*tasks)
    
    print(f"\nFinished processing. Saving to {output_filename}...")
    with open(output_filename, "w", encoding="utf-8") as outfile:
        json.dump(final_results, outfile, indent=2)
        

In [19]:
await run_gemini_pipeline_async("test_easy.json", "gemini_raw_easy.json")

Loading data from test_easy.json...
Firing 524 requests asynchronously...
   [✓] Success: Doc 109
   [✓] Success: Doc 111
   [✓] Success: Doc 198
   [✓] Success: Doc 105
   [✓] Success: Doc 127
   [✓] Success: Doc 99
   [✓] Success: Doc 212
   [✓] Success: Doc 67
   [✓] Success: Doc 130
   [✓] Success: Doc 108
   [✓] Success: Doc 191
   [✓] Success: Doc 88
   [✓] Success: Doc 77
   [✓] Success: Doc 90
   [✓] Success: Doc 213
   [✓] Success: Doc 104
   [✓] Success: Doc 214
   [✓] Success: Doc 100
   [✓] Success: Doc 64
   [✓] Success: Doc 110
   [✓] Success: Doc 268
   [✓] Success: Doc 215
   [✓] Success: Doc 217
   [✓] Success: Doc 263
   [✓] Success: Doc 302
   [✓] Success: Doc 255
   [✓] Success: Doc 265
   [✓] Success: Doc 103
   [✓] Success: Doc 307
   [✓] Success: Doc 305
   [✓] Success: Doc 308
   [✓] Success: Doc 313
   [✓] Success: Doc 312
   [✓] Success: Doc 320
   [✓] Success: Doc 267
   [✓] Success: Doc 368
   [✓] Success: Doc 361
   [✓] Success: Doc 311
   [✓] Success: Doc 

In [20]:
await run_gemini_pipeline_async("test_hard.json", "gemini_raw_hard.json")

Loading data from test_hard.json...
Firing 3616 requests asynchronously...
   [✓] Success: Doc 4
   [✓] Success: Doc 8
   [✓] Success: Doc 9
   [✓] Success: Doc 10
   [✓] Success: Doc 19
   [✓] Success: Doc 17
   [✓] Success: Doc 15
   [✓] Success: Doc 14
   [✓] Success: Doc 28
   [✓] Success: Doc 13
   [✓] Success: Doc 6
   [✓] Success: Doc 0
   [✓] Success: Doc 18
   [✓] Success: Doc 16
   [✓] Success: Doc 12
   [✓] Success: Doc 11
   [✓] Success: Doc 2
   [✓] Success: Doc 39
   [✓] Success: Doc 24
   [✓] Success: Doc 27
   [✓] Success: Doc 45
   [✓] Success: Doc 38
   [✓] Success: Doc 3
   [✓] Success: Doc 50
   [✓] Success: Doc 5
   [✓] Success: Doc 42
   [✓] Success: Doc 49
   [✓] Success: Doc 29
   [✓] Success: Doc 46
   [✓] Success: Doc 48
   [✓] Success: Doc 32
   [✓] Success: Doc 56
   [✓] Success: Doc 35
   [✓] Success: Doc 51
   [✓] Success: Doc 7
   [✓] Success: Doc 66
   [✓] Success: Doc 71
   [✓] Success: Doc 53
   [✓] Success: Doc 47
   [✓] Success: Doc 54
   [✓] Success

In [ ]:
import re
import json

In [ ]:
def extract_spans_for_entity(entity_string, label, original_text):
    spans = []
    if not entity_string.strip():
        return spans
        
    safe_string = re.escape(entity_string)
    
    pattern = rf"(?<!\w){safe_string}(?!\w)"
    
    for match in re.finditer(pattern, original_text, flags=re.IGNORECASE):
        spans.append({
            "label": label,
            "text": original_text[match.start():match.end()], 
            "start": match.start(),
            "end": match.end()
        })
        
    return spans

In [ ]:
def map_document_spans(doc_id, original_text, gemini_response):
    predicted_entities = []
    
    for disease in gemini_response.get("diseases", []):
        found_spans = extract_spans_for_entity(disease, "Disease", original_text)
        predicted_entities.extend(found_spans)
        
    for chemical in gemini_response.get("chemicals", []):
        found_spans = extract_spans_for_entity(chemical, "Chemical", original_text)
        predicted_entities.extend(found_spans)
        
    return {
        "doc_id": doc_id,
        "predicted_entities": predicted_entities
    }

TEST RESULT:
[
  {
    "doc_id": "12345",
    "predicted_entities": [
      {
        "label": "Disease",
        "text": "hepatotoxicity",
        "start": 29,
        "end": 43
      },
      {
        "label": "Disease",
        "text": "fever",
        "start": 81,
        "end": 86
      },
      {
        "label": "Chemical",
        "text": "Acetaminophen",
        "start": 57,
        "end": 70
      }
    ]
  }
]


In [ ]:
def run_full_mapping_pipeline(original_data_file, gemini_raw_file, final_output_file):
    print(f"Loading files: {original_data_file} and {gemini_raw_file}...")
    
    with open(original_data_file, "r", encoding="utf-8") as f:
        original_dataset = json.load(f)
        
    text_lookup = {doc["doc_id"]: doc["text"] for doc in original_dataset}
        
    with open(gemini_raw_file, "r", encoding="utf-8") as f:
        gemini_raw_data = json.load(f)
        
    final_mapped_results = []
    
    for item in gemini_raw_data:
        doc_id = item["doc_id"]
        original_text = text_lookup.get(doc_id, "")
        gemini_resp = item.get("gemini_response", {}) # Added .get() just in case it's missing
        
        mapped_doc = map_document_spans(doc_id, original_text, gemini_resp)
        final_mapped_results.append(mapped_doc)
        
    with open(final_output_file, "w", encoding="utf-8") as f:
        json.dump(final_mapped_results, f, indent=2)
        
    print(f"Success! Mapped spans saved to {final_output_file}")

run_full_mapping_pipeline("test_easy.json", "gemini_raw_easy.json", "gemini_predictions_easy.json")
run_full_mapping_pipeline("test_hard.json", "gemini_raw_hard.json", "gemini_predictions_hard.json")

Loading files: test_easy.json and gemini_raw_easy.json...
Success! Mapped spans saved to gemini_predictions_easy.json
Loading files: test_hard.json and gemini_raw_hard.json...
Success! Mapped spans saved to gemini_predictions_hard.json
